# Local Search Scheduler Example

## Introduction

COASTSim has two schedulers for building a plan ahead of time:

* `PriorityScheduler` places requests in priority order, each at the earliest time
  it fits. It is fast, but it never revisits a decision: once a high-merit
  snapshot is placed, everything after it has to fit around it, and the gaps it
  leaves are filled only by whatever happens to fit them.
* `LocalSearchScheduler` builds the same priority-first plan, then improves it. It
  repeatedly changes the order of the plan's snapshots (inserting, removing,
  swapping or moving one), rebuilds the plan from the new order with the same
  checks the priority scheduler uses, and keeps changes that score well. The
  score is merit-weighted science: each snapshot's merit times its science
  seconds. The plan it returns is never worse than the priority-first one, and
  executes exactly as planned.

This notebook compares the two:

1. on a single one-day plan, with plots of where the time goes;
2. as the search is given more changes to try;
3. in a rolling-horizon simulation, over several random sets of targets.

Local search trades planning time for a better plan, so this notebook takes
several minutes to run, almost all of it in local search.

## Step 1: Import Libraries

In [ ]:
%matplotlib inline
import time
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import numpy as np
from rust_ephem import TLEEphemeris

from conops import DITL, RollingHorizonDITL, create_ditl
from conops.common import ACSMode
from conops.config import (
    MissionConfig,
    ReplanSettings,
    SchedulerConfig,
    Telescope,
    TelescopeConfig,
    TelescopeType,
)
from conops.schedulers import LocalSearchScheduler, PriorityScheduler
from conops.targets import Pointing

## Step 2: Spacecraft Configuration and Ephemeris

The same setup as the Rolling-Horizon DITL example: the default `MissionConfig`
with a small telescope, and a one-day ephemeris from a TLE.

The schedulers work out visibility, slews and ground passes from the ephemeris on
the configuration's constraint, so we attach it there.

In [ ]:
config = MissionConfig()
config.payload.instruments.append(
    Telescope(
        name="Telescope f/7 0.5m",
        optics=TelescopeConfig(
            aperture_m=0.5,
            focal_length_m=3.5,
            telescope_type=TelescopeType.RITCHEY_CHRETIEN,
            tube_length_m=1.0,
        ),
    )
)
config.constraint.ignore_roll = True
config.spacecraft_bus.attitude_control.slew_before_visible = False

begin = datetime(2025, 12, 1, tzinfo=timezone.utc)
end = begin + timedelta(days=1)
ephemeris = TLEEphemeris(begin=begin, end=end, step_size=60, tle="example.tle")
config.constraint.ephem = ephemeris

## Step 3: Create the Targets

100 targets at random positions, each needing 2000 s of exposure in snapshots of
300 to 1000 s, with merit between 10 and 100. That asks for about 55 hours of
exposure in a 24-hour day, so the schedulers have to choose.

The schedulers don't modify the targets they are given, but a simulation credits
collected science to them. `make_targets` builds a fresh set from a random seed,
so every run starts from the same targets.

In [ ]:
def make_targets(seed: int) -> list[Pointing]:
    rng = np.random.default_rng(seed)
    targets = []
    for i in range(100):
        target = Pointing(
            config=config,
            ra=rng.uniform(0, 360),
            dec=rng.uniform(-90, 90),
            obsid=10000 + i,
            name=f"Target {i}",
            merit=rng.uniform(10, 100),
            ss_min=300,
            ss_max=1000,
        )
        target.exptime = 2000
        targets.append(target)
    return targets


targets = make_targets(5)

## Step 4: Build a One-Day Plan with Each Scheduler

`LocalSearchScheduler` takes the same arguments as `PriorityScheduler`, plus settings
for the search. The main ones:

* `time_limit`: seconds to search for (default 10).
* `max_iterations`: the most changes to try. With a fixed `seed`, a fixed number
  of changes gives the same plan every time, whatever the speed of the machine.
* `seed`: the random seed for choosing changes.

Here we let it try 2000 changes.

After planning, `initial_score` is the score of the priority-first plan and
`score` that of the plan returned, both in merit-weighted science seconds.
`iterations` and `accepted` count the changes tried and kept.

In [ ]:
start = time.perf_counter()
priority_scheduler = PriorityScheduler(config, targets, begin, end)
priority_plan = priority_scheduler.schedule()
priority_seconds = time.perf_counter() - start

start = time.perf_counter()
local_scheduler = LocalSearchScheduler(
    config, targets, begin, end, max_iterations=2000, time_limit=300, seed=1
)
local_plan = local_scheduler.schedule()
local_seconds = time.perf_counter() - start

print(f"PriorityScheduler:    {priority_seconds:5.1f} s")
print(
    f"LocalSearchScheduler: {local_seconds:5.1f} s, "
    f"{local_scheduler.iterations} changes tried, {local_scheduler.accepted} kept"
)
print(
    f"Merit-weighted science: {local_scheduler.initial_score / 3600:.0f} -> "
    f"{local_scheduler.score / 3600:.0f} merit-hours"
)

## Step 5: Execute Both Plans

Each plan is executed by a `DITL` with the same configuration, ephemeris, horizon
and step size it was built with. `validate_plan_matches_execution()` returns an
empty list when the simulation did exactly what the plan said.

In [ ]:
def execute(plan, scheduler) -> DITL:
    ditl = DITL(config=config, plan=plan, ephem=ephemeris, begin=begin, end=end)
    ditl.step_size = scheduler.ctx.step_size
    ditl.calc()
    return ditl


priority_ditl = execute(priority_plan, priority_scheduler)
local_ditl = execute(local_plan, local_scheduler)

print(
    "PriorityScheduler mismatches:   ", priority_ditl.validate_plan_matches_execution()
)
print("LocalSearchScheduler mismatches:", local_ditl.validate_plan_matches_execution())

## Step 6: Compare the Results

For each run:

* **science**: hours of science collected;
* **merit-weighted**: each observation's merit times its science hours, summed,
  which is what both schedulers try to maximize;
* **idle**: the share of the day the spacecraft spent idle;
* **observations**, **targets observed** and **targets completed**.

In [ ]:
def science_hours(run) -> float:
    return sum(hk.collection_seconds for hk in run.telemetry.housekeeping) / 3600


def merit_hours(run) -> float:
    science = [e for e in run.plan if e.obstype.name == "AT"]
    seconds = sum(
        e.merit * e.collection_seconds_between(e.begin, e.end) for e in science
    )
    return seconds / 3600


def idle_percent(run) -> float:
    return 100 * np.mean(np.asarray(run.mode) == ACSMode.IDLE)


def seconds_by_target(run) -> dict[int, float]:
    seconds: dict[int, float] = {}
    for e in run.plan:
        if e.obstype.name == "AT":
            collected = e.collection_seconds_between(e.begin, e.end)
            seconds[e.obsid] = seconds.get(e.obsid, 0.0) + collected
    return seconds


print(
    f"{'scheduler':<21} {'science':>8} {'merit-weighted':>15} {'idle':>6} "
    f"{'observations':>12} {'observed':>8} {'completed':>9}"
)
for label, run in (
    ("PriorityScheduler", priority_ditl),
    ("LocalSearchScheduler", local_ditl),
):
    per_target = seconds_by_target(run)
    observations = sum(1 for e in run.plan if e.obstype.name == "AT")
    completed = sum(1 for s in per_target.values() if s >= 2000)
    print(
        f"{label:<21} {science_hours(run):7.2f}h {merit_hours(run):15.0f} "
        f"{idle_percent(run):5.1f}% {observations:12d} {len(per_target):8d} "
        f"{completed:9d}"
    )

## Step 7: Where the Time Goes

The share of the day the spacecraft spent in each attitude-control mode. Local
search turns idle time into science; the time spent slewing barely changes.

In [ ]:
MODE_COLORS = {
    ACSMode.SCIENCE: "tab:green",
    ACSMode.SLEWING: "tab:blue",
    ACSMode.PASS: "tab:orange",
    ACSMode.SAA: "tab:purple",
    ACSMode.CHARGING: "tab:olive",
    ACSMode.SAFE: "black",
    ACSMode.IDLE: "tab:red",
}

runs = {"PriorityScheduler": priority_ditl, "LocalSearchScheduler": local_ditl}

fig, ax = plt.subplots(figsize=(9, 2.5))
for row, run in enumerate(runs.values()):
    modes = np.asarray(run.mode)
    left = 0.0
    for mode, color in MODE_COLORS.items():
        share = 100 * np.mean(modes == mode)
        if share == 0:
            continue
        ax.barh(
            row, share, left=left, color=color, label=mode.name if row == 0 else None
        )
        if share > 3:
            ax.text(
                left + share / 2,
                row,
                f"{share:.1f}%",
                ha="center",
                va="center",
                color="white",
                fontsize=9,
            )
        left += share
ax.set_yticks(range(len(runs)), list(runs))
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Share of the day (%)")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.45), frameon=False)
fig.tight_layout()
plt.show()

## Step 8: The Two Timelines

The spacecraft's mode through the day for each plan, with hours 14 to 18
enlarged below. Red marks idle time. Local search fills many of the short idle
gaps, but the long one a little after 8 hours stays idle in both plans.

In [ ]:
def mode_runs(run):
    """Yield (mode, start hour, length in hours) for each run of one mode."""
    modes = np.asarray(run.mode)
    hours = (np.asarray(run.utime) - run.utime[0]) / 3600
    step = run.step_size / 3600
    changes = np.flatnonzero(np.diff(modes)) + 1
    for first, last in zip(np.r_[0, changes], np.r_[changes, len(modes)]):
        yield ACSMode(modes[first]), hours[first], (last - first) * step


def plot_timelines(ax, xlim):
    for row, run in enumerate(runs.values()):
        for mode, start, length in mode_runs(run):
            ax.broken_barh([(start, length)], (row - 0.4, 0.8), color=MODE_COLORS[mode])
    ax.set_yticks(range(len(runs)), list(runs))
    ax.invert_yaxis()
    ax.set_xlim(*xlim)
    ax.set_xlabel("Hours since start of day")


fig, (full, zoom) = plt.subplots(2, 1, figsize=(10, 4.5))
plot_timelines(full, (0, 24))
plot_timelines(zoom, (14, 18))
full.set_title("Whole day")
zoom.set_title("Hours 14 to 18")
used = [m for m in MODE_COLORS if any(m in run.mode for run in runs.values())]
handles = [plt.Rectangle((0, 0), 1, 1, color=MODE_COLORS[m]) for m in used]
fig.legend(
    handles, [m.name for m in used], ncol=len(used), loc="lower center", frameon=False
)
fig.tight_layout(rect=(0, 0.06, 1, 1))
plt.show()

## Step 9: Merit-Weighted Science Through the Day

The running total of merit-weighted science for each plan (top), and how far
local search is ahead (bottom). Local search is not ahead all day: it reorders
observations, and here falls behind for a few hours in the morning to fit more
in later. It is the total over the plan that it improves.

In [ ]:
def cumulative_merit(run, hours):
    total = np.zeros_like(hours)
    t0 = run.utime[0]
    for e in run.plan:
        if e.obstype.name != "AT":
            continue
        for i, h in enumerate(hours):
            total[i] += e.merit * e.collection_seconds_between(
                e.begin, min(e.end, t0 + h * 3600)
            )
    return total / 3600


hours = np.linspace(0, 24, 289)
curves = {label: cumulative_merit(run, hours) for label, run in runs.items()}

fig, (top, bottom) = plt.subplots(
    2, 1, figsize=(9, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]}
)
for label, curve in curves.items():
    top.plot(hours, curve, label=label)
top.set_ylabel("Merit-weighted science\n(merit-hours)")
top.legend()
bottom.plot(
    hours,
    curves["LocalSearchScheduler"] - curves["PriorityScheduler"],
    color="tab:green",
)
bottom.axhline(0, color="grey", lw=0.8)
bottom.set_ylabel("Local search ahead by")
bottom.set_xlabel("Hours since start of day")
bottom.set_xlim(0, 24)
fig.tight_layout()
plt.show()

## Step 10: Which Targets Gained

The change in planned exposure for each target, from the priority-first plan to
the local search plan, with targets in order of merit. Local search moves time
between targets when that fits more science in overall, so some targets lose a
few minutes. Here most of the changes are among the highest-merit targets, and
the time it finds lets a few lower-merit targets in as well.

In [ ]:
priority_by_target = seconds_by_target(priority_ditl)
local_by_target = seconds_by_target(local_ditl)
by_merit = sorted(targets, key=lambda t: t.merit, reverse=True)
change = np.array(
    [
        local_by_target.get(t.obsid, 0.0) - priority_by_target.get(t.obsid, 0.0)
        for t in by_merit
    ]
)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(
    range(len(by_merit)),
    change / 60,
    color=np.where(change >= 0, "tab:green", "tab:red"),
)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("Targets, highest merit first")
ax.set_ylabel("Change in exposure (min)")
ax.set_xlim(-1, len(by_merit))
fig.tight_layout()
plt.show()

print(f"Targets gaining time: {np.sum(change > 0)}, losing time: {np.sum(change < 0)}")

## Step 11: How Much Search Is Enough?

We build the plan again with different numbers of changes. With
`max_iterations=0` local search returns the priority-first plan unchanged. The
score rises quickly at first and then levels off, while the planning time keeps
growing, so a modest budget gets most of the gain.

In operations `time_limit` is usually the more useful bound, since it caps how
long each plan takes whatever the machine. `max_iterations` is used here so the
results are the same on every machine.

In [ ]:
budgets = [0, 250, 500, 1000, 2000, 4000]
budget_scores = []
budget_seconds = []
for budget in budgets:
    start = time.perf_counter()
    scheduler = LocalSearchScheduler(
        config, targets, begin, end, max_iterations=budget, time_limit=600, seed=1
    )
    scheduler.schedule()
    budget_seconds.append(time.perf_counter() - start)
    budget_scores.append(scheduler.score / 3600)
    print(
        f"{budget:5d} changes: {budget_scores[-1]:6.0f} merit-hours "
        f"in {budget_seconds[-1]:5.1f} s ({scheduler.accepted} kept)"
    )

In [ ]:
fig, (score_ax, time_ax) = plt.subplots(1, 2, figsize=(10, 3.5))
score_ax.plot(budgets, budget_scores, "o-")
score_ax.axhline(budget_scores[0], color="grey", ls="--", label="PriorityScheduler")
score_ax.set_xlabel("Changes tried (max_iterations)")
score_ax.set_ylabel("Merit-weighted science (merit-hours)")
score_ax.legend()
time_ax.plot(budgets, budget_seconds, "o-", color="tab:orange")
time_ax.set_xlabel("Changes tried (max_iterations)")
time_ax.set_ylabel("Planning time (s)")
fig.tight_layout()
plt.show()

## Step 12: Benchmark in Rolling-Horizon Simulations

A single plan is only part of the story: in operations the plan is rebuilt as
the day goes on. Here we run the Rolling-Horizon DITL example's setup (12-hour
horizon, replanning every 6 hours, 15-minute commit lead time, and a ToO 8 hours
in) with each scheduler, on three random sets of targets.

Each run is set up through the configuration's `scheduler` section: rolling mode,
the scheduler `kind` and its settings, and the replanning schedule.
`create_ditl` builds the `RollingHorizonDITL` it describes. The plots show idle time and planning time for each scheduler,
and the science and merit-weighted science local search gains over the
priority-first scheduler.

In [ ]:
# The Rolling-Horizon DITL example's replanning schedule.
REPLANNING = ReplanSettings(
    horizon_seconds=12 * 3600,
    replan_interval_seconds=6 * 3600,
    commit_lead_time_seconds=15 * 60,
    rapid_replans=True,
    allow_interrupts=True,
)


def run_rolling(seed: int, scheduler: SchedulerConfig) -> RollingHorizonDITL:
    config.scheduler = scheduler
    ditl = create_ditl(config, make_targets(seed), begin, end)
    ditl.submit_too(
        obsid=900001,
        ra=150.0,
        dec=30.0,
        merit=1000,
        exptime=600,
        name="GRB 251201A",
        submit_time=begin + timedelta(hours=8),
        deadline=begin + timedelta(hours=8, minutes=45),
    )
    ditl.calc()
    return ditl


SCHEDULERS = {
    "PriorityScheduler": SchedulerConfig(mode="rolling", replanning=REPLANNING),
    "LocalSearchScheduler": SchedulerConfig(
        mode="rolling",
        kind="local_search",
        max_iterations=2000,
        time_limit=300,
        seed=1,
        replanning=REPLANNING,
    ),
}
seeds = [1, 2, 3]
results = {label: [] for label in SCHEDULERS}
for seed in seeds:
    for label, scheduler in SCHEDULERS.items():
        ditl = run_rolling(seed, scheduler)
        results[label].append(
            {
                "idle": idle_percent(ditl),
                "science": science_hours(ditl),
                "merit": merit_hours(ditl),
                "planning": sum(r.planning_seconds for r in ditl.replans),
                "mismatches": len(ditl.validate_plan_matches_execution()),
            }
        )
        r = results[label][-1]
        print(
            f"seed {seed}  {label:<21} idle {r['idle']:4.1f}%  science {r['science']:5.2f} h  "
            f"merit-weighted {r['merit']:5.0f}  planning {r['planning']:5.1f} s  "
            f"mismatches {r['mismatches']}"
        )

In [ ]:
METRICS = {
    "idle": "Idle (% of day)",
    "science": "Science (h)",
    "merit": "Merit-weighted science\n(merit-hours)",
    "planning": "Planning time (s)",
}

priority_rows = results["PriorityScheduler"]
local_rows = results["LocalSearchScheduler"]
x = np.arange(len(seeds))
labels = [f"seed {s}" for s in seeds]
width = 0.38

fig, (idle_ax, science_ax, merit_ax, time_ax) = plt.subplots(1, 4, figsize=(13, 3.5))

# Idle time and planning time side by side for the two schedulers.
for ax, key in ((idle_ax, "idle"), (time_ax, "planning")):
    for offset, (label, rows) in zip((-width / 2, width / 2), results.items()):
        ax.bar(x + offset, [r[key] for r in rows], width, label=label)
    ax.set_title(METRICS[key])
idle_ax.legend(fontsize=8)

# Science and merit-weighted science as the gain from local search, which is
# small next to the totals.
science_gain = [
    ls["science"] - pr["science"] for pr, ls in zip(priority_rows, local_rows)
]
merit_gain = [
    100 * (ls["merit"] / pr["merit"] - 1) for pr, ls in zip(priority_rows, local_rows)
]
science_ax.bar(x, science_gain, color="tab:green")
science_ax.set_title("Science gained (h)")
merit_ax.bar(x, merit_gain, color="tab:green")
merit_ax.set_title("Merit-weighted science gained (%)")

for ax in (idle_ax, science_ax, merit_ax, time_ax):
    ax.set_xticks(x, labels)
fig.tight_layout()
plt.show()

print(f"{'mean':<21} {'idle':>6} {'science':>8} {'merit-weighted':>15} {'planning':>9}")
for label, rows in results.items():
    mean = {key: np.mean([r[key] for r in rows]) for key in METRICS}
    print(
        f"{label:<21} {mean['idle']:5.1f}% {mean['science']:7.2f}h "
        f"{mean['merit']:15.0f} {mean['planning']:8.1f}s"
    )

## Summary

* `LocalSearchScheduler` starts from the plan `PriorityScheduler` would build and
  only keeps it if it can't find a better one, so it is never worse.
* Every plan it considers is checked the same way as a priority-first plan, so
  it executes exactly as planned.
* On these targets it cuts idle time and adds science and merit-weighted science,
  in a single plan and in a rolling-horizon simulation.
* The cost is planning time. Bound it with `time_limit` (seconds per plan) or
  `max_iterations` (changes per plan); most of the gain comes early.

Other settings:

* `earliness_weight` (default 0.5): how much a request with a deadline loses for
  being planned later, so time-critical requests are not pushed later than they
  need to be.
* `neighborhood` (default 3): how far apart in the sequence two snapshots can be
  to be swapped or moved.
* `history_length` (default 50): a change is kept if it is no worse than the
  current plan, or than the plan this many steps ago. A longer history lets the
  search accept more changes that cost a little on the way to better plans.

See the planning guide (Improving plans by local search) for details.